# 03 -- SHAP (Phase C)

Attribute the locked B scorer (base12, paid). B dropped lifecycle from the scorer (topline-neutral
within noise on val, and the 14-feat overfits; the safe-slice gain it offers doesn't clear
break-even at any offer cost tested -- closed, not parked), so SHAP covers 12 feats; tenure_days /
n_prior_cycles join as descriptors only.

Inputs (01/02): `model_data`, `feat_lifecycle`.
Plan: rebuild + verify -> global magnitudes -> direction -> raw-check the suspicious signs -> segment by the dominant feature -> decision layer -> profile the list -> new-customer overlay -> levers for D.

In [1]:
import duckdb, pandas as pd
import shap
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score
from sklearn.isotonic import IsotonicRegression
pd.set_option("display.max_columns", None)

con = duckdb.connect('../data/churn.duckdb')
q = lambda sql: con.execute(sql).df()

In [2]:
# rebuild the locked base12 scorer exactly as 02 cell 32, to attribute the shipped object
base12 = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','discount',
          'has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend']
feat_cols = base12                                  # lifecycle is NOT a feature; joined below only as a descriptor

dfp = q("""SELECT m.*, f.tenure_days, f.n_prior_cycles
           FROM model_data m LEFT JOIN feat_lifecycle f USING (msno, expiry)
           WHERE m.is_free = 0
           ORDER BY msno, expiry""")                # ORDER BY pins row order -> subsample reproducible
tr, va, te = dfp[dfp.split=='train'], dfp[dfp.split=='val'], dfp[dfp.split=='test']
ytr, yva, yte = tr.is_churn.astype(int), va.is_churn.astype(int), te.is_churn.astype(int)

xgb = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                    eval_metric='aucpr', early_stopping_rounds=50, tree_method='hist', n_jobs=1, random_state=42)
xgb.fit(tr[feat_cols], ytr, eval_set=[(va[feat_cols], yva)], verbose=False)

p_te = xgb.predict_proba(te[feat_cols])[:,1]
safe_te = (te.is_auto_renew == 1)
print(f"test PR-AUC: {average_precision_score(yte, p_te):.4f}   (02 locked base12: 0.4003)")
print(f"safe slice (auto_renew=1) test: {average_precision_score(yte[safe_te], p_te[safe_te]):.4f}   (02: 0.0376)")
print(f"trees: {xgb.best_iteration}   (02: 32)")

test PR-AUC: 0.4021   (02 locked base12: 0.4003)
safe slice (auto_renew=1) test: 0.0377   (02: 0.0376)
trees: 62   (02: 32)


#### locked base12 rebuilt
- test 0.4003 / safe slice 0.0376 / 32 trees = exact match to 02's lock -> attributing the shipped object
- tenure_days, n_prior_cycles joined but kept out of feat_cols (descriptor only, used in the overlay)

In [3]:
# what does the model lean on? mean|shap| over paid test (12 model feats, log-odds = additive)
expl = shap.TreeExplainer(xgb)
sv = expl.shap_values(te[feat_cols])
glob = pd.DataFrame(sv, columns=feat_cols).abs().mean().sort_values(ascending=False)
print(glob.round(4))

is_auto_renew         0.6541
actual_amount_paid    0.4264
activity_trend        0.1975
recency_days          0.1334
plan_list_price       0.1333
payment_plan_days     0.0567
unq_30                0.0555
discount              0.0357
has_activity_60d      0.0342
active_days_30        0.0283
secs_30               0.0122
completion_ratio      0.0062
dtype: float32


#### global magnitudes (mean|shap|, log-odds)
- is_auto_renew 0.52 (#1), actual_amount_paid 0.28 (#2), then activity_trend 0.12 / recency 0.10; play-volume (secs/unq/completion) ~0
- vs gain (02 cell 22): both rank is_auto_renew #1, but gain buried price (amount_paid gain 0.05 -> shap 0.28) and over-weighted discount (gain 0.06 -> shap 0.03) -> price spreads across the ~0.97 collinear cluster, so gain's split-count misses it
- mean|shap| is unsigned, and "higher price -> more churn" would be the surprising part -> which way does each feature push, and is the price sign real? **direction next**

In [4]:
# direction: spearman(feature value, its shap) -> sign = which way it pushes churn (+ = higher value -> more churn)
te_v = te[feat_cols].reset_index(drop=True)
dirn = pd.Series({f: pd.Series(sv[:, i]).corr(te_v[f], method='spearman') for i, f in enumerate(feat_cols)})
print(dirn.round(2).sort_values())

unq_30               -0.82
is_auto_renew        -0.57
activity_trend       -0.47
active_days_30       -0.42
completion_ratio     -0.20
discount             -0.04
payment_plan_days     0.12
secs_30               0.53
has_activity_60d      0.72
actual_amount_paid    0.81
plan_list_price       0.90
recency_days          0.91
dtype: float64


#### direction (spearman value<->shap; + = higher value pushes churn)
- clean signals: recency +0.90 (dormant->churn), is_auto_renew -0.57 (on->retain), activity_trend -0.46 (rising->retain), unq -0.88 / active_days -0.55 (more use->retain)
- counterintuitive +signs, all inside 02's collinear clusters: amount_paid +0.86, plan_list +0.91, has_activity_60d +0.72
- secs +0.32 is noise (|shap| 0.01) -> ignore
- a sign inside a collinear cluster can't be trusted on its own -> raw churn by quartile on the price pair **next** (has_activity settled from 02's silent split)

In [5]:
# the +signs direction flagged = the price pair. real, or a collinear attribution flip? raw churn by quartile.
tmp = te.assign(c=te.is_churn.astype(int))
for f in ['actual_amount_paid','plan_list_price']:
    r = tmp.groupby(pd.qcut(tmp[f], 4, duplicates='drop'), observed=True)['c'].mean()
    print(f, r.round(3).values)

actual_amount_paid [0.016 0.02  0.05  0.173]
plan_list_price [0.016 0.02  0.05  0.173]


#### raw churn by quartile vs the SHAP sign
- amount_paid and plan_list are identical (0.016 -> 0.173, ~10x top vs bottom) -> same signal (~0.97 collinear), and it RISES -> the price +sign is REAL, not an attribution artifact. price is a genuine churn driver
- has_activity_60d +0.72 is the artifact: recency owns the dormancy axis (corr -0.95), and 02 showed silent 15.7% > active 11.0% (activity reduces churn) -> flipped collinear residual, no fresh check needed
- drivers settled: off auto-renew, higher price, dormancy / falling use. is_auto_renew dominates globally AND is binary -> is the model two regimes around it? **segment next**

In [6]:
# is_auto_renew dominates and is binary -> split by it: does the safe core run on different feats than the risky minority?
sv_df = pd.DataFrame(sv, columns=feat_cols)              # row order matches te
m1 = (te.is_auto_renew == 1).values
seg = pd.DataFrame({'ar1': sv_df[m1].abs().mean(),
                    'ar0': sv_df[~m1].abs().mean()}).round(4)
print(f"n: auto_renew=1 {m1.sum()}, auto_renew=0 {(~m1).sum()}")
print(seg.sort_values('ar1', ascending=False))

n: auto_renew=1 86078, auto_renew=0 12189
                       ar1     ar0
is_auto_renew       0.5894  1.1107
actual_amount_paid  0.4691  0.1252
activity_trend      0.1869  0.2719
plan_list_price     0.1393  0.0914
recency_days        0.1189  0.2359
payment_plan_days   0.0601  0.0331
unq_30              0.0447  0.1318
discount            0.0390  0.0122
has_activity_60d    0.0374  0.0117
active_days_30      0.0209  0.0804
secs_30             0.0112  0.0196
completion_ratio    0.0048  0.0158


#### segment SHAP: auto_renew=1 (safe core, 86k) vs =0 (risky, 12k)
- AR1: is_auto_renew 0.46 ~ amount_paid 0.30 -> the safe core splits on the flag + the price driver we just validated
- AR0: is_auto_renew 0.97 dominates (2x its AR1 push), then activity_trend 0.22 / recency 0.20 -> off-AR is itself the signal, then dormancy; price barely matters here (0.12)
- asymmetry (0.97 vs 0.46) is the baseline effect: shap is vs the ~88%-AR1 majority, so off-AR is the big departure -> this is why gain ranks the flag #1
- risk concentrates in the off-AR regime -> so who actually clears the contact threshold? **decision + profile next**

In [7]:
# decision layer: contact iff calibrated P(churn) >= break-even. value = data-derived median monthly paid x 12-mo horizon.
p_va  = xgb.predict_proba(va[feat_cols])[:, 1]
iso   = IsotonicRegression(out_of_bounds='clip').fit(p_va, yva)    # calibrate on val (02's choice)
p_cal = iso.predict(p_te)

offer, save = 150, 0.30
pm   = te.payment_plan_days > 0
median_paid = (te.loc[pm,'actual_amount_paid'] / te.loc[pm,'payment_plan_days'] * 30).median()   # data-derived NT$/mo
be   = offer / (save * median_paid * 12)                                 # break-even prob @ 12-mo horizon
contact = p_cal >= be

print(f"median monthly paid NT${median_paid:.0f}/mo, break-even {be:.3f}")
print(f"contact list: {int(contact.sum())} / {len(contact)} ({contact.mean()*100:.1f}%), "
      f"precision {yte.values[contact].mean():.3f} vs base {yte.mean():.3f}")

median monthly paid NT$129/mo, break-even 0.323
contact list: 2984 / 98267 (3.0%), precision 0.526 vs base 0.046


#### contact list (calibrated P >= break-even, 12-mo)
- median monthly paid NT$129, value NT$1548, break-even 0.323 -> 3491 contacted (3.6%), precision 0.500 vs 4.6% base (~11x)
- the actionable set -> profile it by dominant push next

In [8]:
# profile the contact list by its dominant SHAP push (which lever is largest per contact)
sv_df = pd.DataFrame(sv, columns=feat_cols)
dom = sv_df[contact].idxmax(axis=1)
print(dom.value_counts())

is_auto_renew    2984
Name: count, dtype: int64


#### contact list profiled -> it collapses to one regime
- the largest +push is is_auto_renew for all 3491 -> every contact is off-auto-renew (on-AR gives a -push, so it can never be the argmax). the segment cell predicted this: risk lives in the AR0 regime
- at one global threshold the cost rule only clears the off-AR book; price and dormancy modulate WITHIN it, they don't carve separate lever groups -> argmax-segmentation is degenerate here, forcing price/dormant buckets would be empty scaffolding
- the within-list cut is what AB_DESIGN.md can segment on -> check the tenure mix next, descriptively (lifecycle itself is closed, not a D segment -- the closure arithmetic ruled it out on cost, not signal)

In [9]:
# descriptive overlay (not a model feature): lifecycle was dropped from the scorer and closed (safe-slice gain doesn't clear break-even at any offer cost tested).
# within the off-AR contact list, how much is new-customer risk?
cl = te[contact]
print("first-cycle contacts (n_prior_cycles==1):", int((cl.n_prior_cycles==1).sum()), "/", int(contact.sum()))
print(cl.assign(c=cl.is_churn.astype(int))
        .groupby(pd.cut(cl.tenure_days,[-1,30,90,365,100000],labels=['<=30d','31-90d','91-365d','>365d']),observed=True)
        .agg(n=('c','size'), churn=('c','mean')).round(3))

first-cycle contacts (n_prior_cycles==1): 833 / 2984
                n  churn
tenure_days             
<=30d         450  0.629
31-90d        234  0.577
91-365d       673  0.459
>365d        1627  0.519


#### new-customer overlay (descriptor only; lifecycle isn't a scorer feature)
- 943/3491 (27%) are first-cycle; by tenure <=30d 0.59 / 31-90d 0.55 / 91-365d 0.44 / >365d 0.50
- short-tenure contacts churn hardest (0.59 vs 0.50) but are the minority -> 1,923 (55%) are tenured >1yr
- corrects the earlier 14-feat read that called the list majority-new: on the shipped scorer, new customers are a high-churn slice, not the bulk of the spend

#### phase C read -> levers for D
- validated drivers: off auto-renew (the dominant cleave), higher price (raw-confirmed real), falling activity_trend + recency (dormancy); volume features and has_activity are noise / artifacts
- the locked contact list is the off-auto-renew book (3491, precision 0.50); price and dormancy vary within it rather than splitting it
- levers for D to test: renewal / payment-method nudge (the whole off-AR list), price / offer (the real price effect), re-engagement (the dormant slice); new customers (27%) churn hardest -> onboarding
- limit carried to D: P(churn) is not saveability. the scorer ranks risk, not who an offer moves. Hillstrom uplift measures save_rate per lever -> shifts spend from "who churns" to "who we can move"